# InterScale Pipeline

In [1]:
import scanpy as sc

import InterScale as interscale
from InterScale.config import load_config
from graph_transformer_long_range_niches.pp import split_adata
from InterScale.tl import prepare_geome_dataset
from InterScale.geome_dataloader import GraphAnnDataModule

# from graph_transformer_long_range_niches.config import load_config
# from graph_transformer_long_range_niches.pp import sliding_window

/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Imp

In [2]:
CFG_PATH = "/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/Legnini_23/legnini23_genes_sample_LocalModel_gnn.yaml"

## 0. Load and prepare data

We load a subset of the CosmX pancreas data containing T1D (type 1 diabetes) and ND (no diabetes) samples and the config file with the model specifications. 

In [3]:
cfg = load_config(CFG_PATH)
cfg

CfgNode({'wandb': CfgNode({'use': True, 'project_name': 'GTLongRange_Legnini23'}), 'model': CfgNode({'n_embed': 32, 'local_component': CfgNode({'name': 'GCN', 'load': None, 'parameters': CfgNode({'embed_dim': 32, 'hidden_dim': 128, 'num_layers': 2, 'dropout': 0.0})}), 'global_component': CfgNode({'name': None, 'load': None}), 'save': None, 'loss': 'GaussianNLL', 'decoder': CfgNode({'type': 'linear', 'hidden_dims': [256, 128], 'dropout': 0.1})}), 'optim': CfgNode({'lr': 0.001, 'wd': 0.0, 'lr_warmup': 20, 'loss': 'CrossEntropy', 'seed': 44, 'cross_corr': 'gene', 'n_epochs': 4000}), 'dataset': CfgNode({'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/data/legnini23.h5ad', 'name': 'legnini23', 'description': '', 'prediction_task': 'classification', 'prediction_obs': 'condition', 'prediction_level': 'node', 'layer_key': 'log1p_norm', 'sample_key': ['sample'], 'group_label': 'condition', 'split_key': 'split', 'spatial_neigbors_kwargs': CfgNode({'radius'

In [4]:
DATA_PATH = '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/data/legnini23.h5ad'

In [5]:
adata = sc.read_h5ad(DATA_PATH)
adata

AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names'
    var: 'gene_ids', 'feature_types'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

If the samples are too large (more cells per sample than the transformers context length) then split the data into sliding windows. 

In [ ]:
sample_key = 'fov'

if adata.obs['fov'].mean() > cfg.transformer.context_length:
    SLIDING_WINDOW_KEY = 'sliding_window_square'
    sliding_window(
        adata,
        library_key = 'slide_fov',
        window_size = None,
        overlap = 0,
        max_n_cells=cfg.transformer.context_length,
        partial_window = 'merge',
        square = True,
        sliding_window_key = SLIDING_WINDOW_KEY,
        copy = False,
    ) 
    sample_key = SLIDING_WINDOW_KEY

Build a spatial neighborhood graph using `squidpy`.

In [ ]:
assert cfg.dataset.spatial_neigbors_kwargs.radius

## 1. Data setup

We need to specify:

- `prediction_task`: Prediction task can either be classification or regression
- `prediction_level`: Which level the predictions should be performed on: either (1) tissue label, e.i. condition (`graph`), (2) node label (`node`) for cell type or niche prediciton, or (3) GEX prediction.

Additionally, we define dataset specific keys: 
- `prediction_obs`: Label in `adata.obs` to be predicted. Only required for classification tasks.
- `layer_key`: Defines which GEX matrix to retrieve from `adata.layer`
- `sample_key`: `adata.obs` used to split the samples into PyG Data objects
- `group_label`: Optional: only if we have a `adata.obs` group that we want to stratify during sampling

In [6]:
PREDICTION_TASK = 'classification'
PREDICTION_LEVEL = 'graph'

prediction_obs = 'condition'
layer_key = 'log1p_norm'
sample_key = 'sample'
group_label = 'condition'

In [7]:
interscale.model.LocalModel._setup_anndata(adata = adata, prediction_task = PREDICTION_TASK, layer_key = layer_key, sample_key = sample_key, prediction_obs = group_label)

/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[log1p_norm] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)


Anndata setup with scvi-tools version 1.3.0.

     Summary Statistics     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Summary Stat Key ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ n_prediction_obs │   2   │
│   n_sample_key   │  17   │
│       n_x        │  88   │
└──────────────────┴───────┘

                    Data Registry                     
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  Registry Key  ┃        scvi-tools Location        ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prediction_obs │ adata.obs['_scvi_prediction_obs'] │
│   sample_key   │   adata.obs['_scvi_sample_key']   │
│       x        │    adata.layers['log1p_norm']     │
└────────────────┴───────────────────────────────────┘

                prediction_obs State Registry                
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃    Source Location     ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['condition'] │    Ctrl    │          0          │
│                        │    SHH     │          1          │
└────────────────────────┴────────────┴─────────────────────┘

                 sample_key State Registry                 
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃   Source Location   ┃ Categories  ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['sample'] │ slide1_A2-1 │          0          │
│                     │ slide1_A2-2 │          1          │
│                     │ slide1_B2-1 │          2          │
│                     │ slide1_B2-2 │          3          │
│                     │ slide1_B2-3 │          4          │
│                     │ slide1_C2-1 │          5          │
│                     │ slide1_C2-2 │          6          │
│                     │ slide1_C2-3 │          7          │
│                     │ slide1_C2-5 │          8          │
│                     │ slide1_D2-2 │          9          │
│                     │ slide1_D2-3 │         10          │
│                     │ slide4_A2-1 │         11          │
│                     │ slide4_A2-2 │         12          │
│                     │ slide4_A2-3 │         13          │
│                     │ slide4_B2-1 │         14          │
│                     │ slide4_B2-2 │         15          │
│                     │ slide4_B2-3 │         16          │
└─────────────────────┴─────────────┴─────────────────────┘

In [8]:
adata

AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names', '_scvi_prediction_obs', '_scvi_sample_key'
    var: 'gene_ids', 'feature_types'
    uns: '_scvi_uuid', '_scvi_manager_uuid'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

## 2. Model setup

In [9]:
model = interscale.model.LocalModel(
    adata,
    prediction_task = PREDICTION_TASK,
    cfg = cfg
)

In [10]:
model._model_summary_string

'Local compnent GCN: n_layers: 2,n_hidden: 128,n_embed: 32, dropout_rate: 0.0'

## 3. Training

In [11]:
split_adata(adata, split_obs='sample', val_size=cfg.dataset.val_size, test_size=cfg.dataset.test_size, seed = cfg.optim.seed, stratify_groups = cfg.dataset.group_label)
pyg_data_list, _ = prepare_geome_dataset(adata, cfg)
dm = GraphAnnDataModule(datas=pyg_data_list, 
                           num_workers=1, 
                           batch_size=int(cfg.dataset.batch_size), 
                           pct_mask_nodes=cfg.dataset.pct_mask_nodes,
                           learning_type="node")

Stratifying by conditions: ['Ctrl' 'SHH']
Test size > 0
{'counts': {'train': 29267, 'val': 10732, 'test': 3763}, 'groups': {'train': ['slide1_A2-2', 'slide4_B2-3', 'slide1_C2-1', 'slide1_D2-3', 'slide4_B2-1', 'slide1_B2-2', 'slide4_A2-1', 'slide1_C2-3', 'slide1_B2-1', 'slide1_D2-2', 'slide4_A2-2'], 'val': ['slide1_C2-2', 'slide1_B2-3', 'slide1_C2-5', 'slide1_A2-1'], 'test': ['slide4_A2-3', 'slide4_B2-2']}}
Split key split already exists in adata.obs
coord_type: generic
library_key: sample
n_neighs: 6
radius: 200
call new


/ictstr01/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:38: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")


{'condition': ['Ctrl', 'SHH']}
call new
{'condition': ['Ctrl', 'SHH']}


/ictstr01/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:38: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")


call new
{'condition': ['Ctrl', 'SHH']}
call new
{'condition': ['Ctrl', 'SHH']}
Masked dataloader


/ictstr01/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:38: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")
/ictstr01/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:38: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")


In [12]:
# for now training only works with datamodule. TODO: Add DataSplitter for datamodule independent training.
model.train(max_epochs = 10, 
           datamodule = dm,
           early_stopping = True)

AttributeError: 'TrainingPlan' object has no attribute '_cfg'

## 4. Evaluation

Evaluation can either be run on the entire AnnData that the model was set up with or subsets of AnnData defined by sample_id from `adata.obs[sample_key]`.

In [ ]:
embedding = model.local_component.get_embedding(sample_id = [],
                                save_results = False)